# Bab 3: Pemodelan & Evaluasi
Dokumen ini berisi proses eksperimen, perbandingan algoritma (Random Forest vs SVM), pencarian hyperparameter terbaik, hingga evaluasi performa model untuk klasifikasi hand gesture.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

## 1. Memuat Dataset yang Telah Dibersihkan (Clean Dataset)

Pada tahap ini, kita memuat file `hand_landmarks_clean.csv` dari folder `data/processed/`. Data kemudian dipisahkan menjadi fitur (`X`) yang berisi koordinat tangan dan target (`y`) yang berisi label gerakan tangan.

In [ ]:
# Membaca dataset yang berada di folder data/processed
# Jalur mundurnya disesuaikan karena notebook berada di dalam folder notebooks/
data_path = '../data/processed/hand_landmarks_data_clean.csv'

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"[+] Dataset berhasil dimuat! Ukuran data: {df.shape}")
else:
    print("[-] File dataset tidak ditemukan. Pastikan file hand_landmarks_data_clean.csv sudah ada.")

# Pisahkan fitur (X) dan target/label (y)
X = df.drop('label', axis=1)
y = df['label']

[-] File dataset tidak ditemukan. Pastikan file hand_landmarks_data_clean.csv sudah ada.


NameError: name 'df' is not defined

## 2. Pembagian Dataset (Train-Test Split)

Sesuai dengan instruksi Soal 2 Poin 5, dataset dibagi dengan rasio 80% untuk data training (pelatihan) dan 20% untuk data testing (pengujian). Kita menggunakan parameter `stratify=y` agar distribusi kelas di data train dan test tetap seimbang.

In [ ]:
# Sesuai Soal 2 Poin 5: Membagi data menjadi 80% training dan 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"[+] Jumlah Data Training : {X_train.shape[0]} sampel")
print(f"[+] Jumlah Data Testing  : {X_test.shape[0]} sampel")

## 3. Pemodelan 1: Random Forest Classifier + Hyperparameter Tuning

Melatih model pertama menggunakan algoritma Random Forest. Untuk memenuhi kriteria Soal 3 Poin 2, dilakukan proses tuning hyperparameter menggunakan `RandomizedSearchCV` guna mencari kombinasi parameter terbaik (seperti `n_estimators` dan `max_depth`) secara efisien.

In [ ]:
print("[*] Melakukan Tuning Hyperparameter pada Model 1 (Random Forest)...")

# Definisikan ruang pencarian parameter (Hyperparameter Space)
param_dist = {
    'n_estimators': [50, 100, 150],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'criterion': ['gini', 'entropy']
}

# Menggunakan RandomizedSearchCV agar proses running lebih cepat namun optimal
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42), 
    param_distributions=param_dist, 
    n_iter=5, 
    cv=3, 
    random_state=42, 
    n_jobs=-1
)

rf_search.fit(X_train, y_train)
best_rf_model = rf_search.best_estimator_

# Prediksi komponen test
y_pred_rf = best_rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, y_pred_rf)

print(f"[+] Parameter Terbaik RF: {rf_search.best_params_}")
print(f"[+] Akurasi Random Forest setelah Tuning: {rf_acc * 100:.2f}%")

## 4. Pemodelan 2: Support Vector Machine (SVM) sebagai Pembanding

Sesuai instruksi Soal 3 Poin 1 yang mewajibkan minimal 2 model berbeda, kita melatih algoritma kedua yaitu Support Vector Machine (SVM) dengan kernel linear. Model ini akan digunakan sebagai baseline pembanding untuk melihat model mana yang lebih optimal.

In [ ]:
print("[*] Melatih Model 2 (Support Vector Machine) sebagai pembanding...")

# Menggunakan model SVM dengan kernel linear agar komputasi koordinat tangan efisien
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
svm_model.fit(X_train, y_train)

# Prediksi komponen test
y_pred_svm = svm_model.predict(X_test)
svm_acc = accuracy_score(y_test, y_pred_svm)

print(f"[+] Akurasi SVM (Baseline): {svm_acc * 100:.2f}%")

## 5. Tabel Perbandingan Performa Model & Justifikasi Terbaik

Berikut adalah tabel ringkasan metrik akurasi dari kedua model yang diuji (Soal 3 Poin 3). Di akhir cell ini, akan muncul narasi **Justifikasi** untuk menentukan model mana yang paling layak diekspor menjadi file `.pkl` dan digunakan pada aplikasi Streamlit.

In [ ]:
# Membuat ringkasan performa dalam bentuk DataFrame untuk menjawab Soal 3 
performa_summary = pd.DataFrame({
    'Metrik Evaluasi': ['Akurasi (Accuracy)'],
    'Model 1: Random Forest (Tuned)': [f"{rf_acc * 100:.2f}%"],
    'Model 2: SVM (Linear)': [f"{svm_acc * 100:.2f}%"]
})

print("="*60)
print("         TABEL PERBANDINGAN PERFORMA MODEL AI")
print("="*60)
display(performa_summary)
print("="*60)

# Menentukan Justifikasi Model Terbaik [cite: 80, 85]
if rf_acc > svm_acc:
    print(f"JUSTIFIKASI: Model Random Forest dipilih sebagai model terbaik karena unggul dengan akurasi {rf_acc*100:.2f}%. Model ini yang diekspor menjadi file `.pkl`.")
else:
    print(f"JUSTIFIKASI: Model SVM dipilih sebagai model terbaik karena unggul dengan akurasi {svm_acc*100:.2f}%. Model ini yang diekspor menjadi file `.pkl`.")

## 6. Visualisasi Evaluasi (Confusion Matrix) & Laporan Klasifikasi Detail

Langkah terakhir adalah menampilkan visualisasi *Confusion Matrix* menggunakan grafik heatmap Seaborn untuk melihat performa model terbaik dalam memprediksi tiap kelas secara spesifik. Diikuti dengan *Classification Report* (Precision, Recall, F1-Score) untuk evaluasi menyeluruh.

In [ ]:
# Mengambil model terbaik untuk divisualisasikan secara detail
# (Diasumsikan Random Forest yang menang berdasarkan hasil running kamu sebelumnya)
cm = confusion_matrix(y_test, y_pred_rf)
labels = np.unique(y_test)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix - Model Terbaik (Random Forest)', fontsize=14, pad=15)
plt.xlabel('Label Prediksi AI', fontsize=12)
plt.ylabel('Label Asli (Ground Truth)', fontsize=12)
plt.tight_layout()
plt.show()

# Tampilkan laporan detail per kelas/gerakan tangan [cite: 79]
print("\nLaporan Klasifikasi Detail Model Terbaik:")
print(classification_report(y_test, y_pred_rf))